# Polars analysis caches

This notebook treats `trace.csv` and `summary.json` as stable CDT interchange files, then builds local Parquet caches for exploratory run debugging. The Parquet files live under `target/notebooks/analysis-caches/` and are disposable analysis artifacts, not part of the Rust crate API.

## 1. Setup

Run `just notebook-setup` before opening this notebook. The cells below find or build the `cdt` binary, run two moderate deterministic simulations, and keep all generated files under `target/`. This notebook is meant for debugging runs and is linted by CI, but it is heavier than the fast demo notebooks.

In [ ]:
import json
import math
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import TYPE_CHECKING, Any, TypedDict, cast

import matplotlib.pyplot as plt
import polars as pl

if TYPE_CHECKING:
    from collections.abc import Callable

UTF8 = "utf-8"


class SlabTriangleProfileRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    step: int
    slice: int
    triangles: float


class MeasurementRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    step: int
    action: float
    vertices: int
    edges: int
    triangles: int
    slab_triangle_total: float


class RunSummaryRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    steps: int
    thermalization_steps: int
    measurement_frequency: int
    acceptance_rate: float
    average_action: float
    elapsed_time_ms: float
    measurement_count: int
    step_count: int
    final_vertices: int
    final_edges: int
    final_triangles: int
    final_time_slices: int
    average_slab_triangle_total: float
    max_slab_triangle_fluctuation: float
    all_scale_effective_hausdorff_slope: float | None
    short_time_effective_spectral_dimension: float | None
    effective_dimensions_exported: bool


class CounterRow(TypedDict):
    run_id: str
    category: str
    counter: str
    value: int


@dataclass(frozen=True, slots=True)
class RunConfig:
    run_id: str
    seed: int
    topology: str = "open-boundary"
    cosmological_constant: float = 0.46209812037329684
    temperature: float = 1.0
    vertices_per_slice: int = 12
    timeslices: int = 10
    steps: int = 1_000
    thermalization_steps: int = 100
    measurement_frequency: int = 20


@dataclass(frozen=True, slots=True)
class RunArtifacts:
    run_dir: Path
    trace_csv: Path
    summary_json: Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "Cargo.toml").is_file() and (path / "src" / "main.rs").is_file():
            return path
    message = "Run this notebook from inside the causal-triangulations repository."
    raise RuntimeError(message)


def run_command(command: list[str], *, cwd: Path, timeout: int = 300) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(  # noqa: S603 - this notebook intentionally invokes the repository binary with a fixed argv list.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.returncode != 0:
        command_text = " ".join(command)
        message = f"command failed with exit code {result.returncode}: {command_text}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def cdt_binary_path(root: Path) -> str:
    configured = os.environ.get("CDT_BINARY")
    if configured is not None:
        binary = Path(configured).expanduser().resolve()
        if not binary.is_file():
            message = f"CDT_BINARY does not point to a file: {binary}"
            raise FileNotFoundError(message)
        return str(binary)

    binary_name = "cdt.exe" if os.name == "nt" else "cdt"
    local_binary = root / "target" / "release" / binary_name
    run_command(
        ["cargo", "build", "--locked", "--release", "--bin", "cdt"],
        cwd=root,
        timeout=600,
    )
    if not local_binary.is_file():
        message = f"cargo did not produce the expected cdt binary: {local_binary}"
        raise FileNotFoundError(message)
    return str(local_binary)


def artifacts_for(root: Path, config: RunConfig) -> RunArtifacts:
    run_dir = root / "target" / "notebooks" / "analysis-caches" / "runs" / config.run_id
    return RunArtifacts(run_dir=run_dir, trace_csv=run_dir / "trace.csv", summary_json=run_dir / "summary.json")


ROOT = find_repo_root(Path.cwd().resolve())
CDT_BINARY = cdt_binary_path(ROOT)
CACHE_DIR = ROOT / "target" / "notebooks" / "analysis-caches" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUNS = (
    RunConfig(run_id="seed-105", seed=105),
    RunConfig(run_id="seed-106", seed=106, cosmological_constant=0.5),
)

print(f"Repository: {ROOT}")
print(f"Using cdt binary: {CDT_BINARY}")



## 2. Generate stable interchange files

The Rust binary writes CSV and JSON. Those files are the stable interchange boundary. Parquet appears only after the notebook reads and reshapes those outputs.

In [ ]:
def run_simulation(config: RunConfig) -> RunArtifacts:
    artifacts = artifacts_for(ROOT, config)
    artifacts.run_dir.mkdir(parents=True, exist_ok=True)
    command = [
        CDT_BINARY,
        "--dimension",
        "2",
        "--vertices-per-slice",
        str(config.vertices_per_slice),
        "--timeslices",
        str(config.timeslices),
        "--topology",
        config.topology,
        "--cosmological-constant",
        str(config.cosmological_constant),
        "--temperature",
        str(config.temperature),
        "--steps",
        str(config.steps),
        "--thermalization-steps",
        str(config.thermalization_steps),
        "--measurement-frequency",
        str(config.measurement_frequency),
        "--seed",
        str(config.seed),
        "--simulate",
        "--output-csv",
        str(artifacts.trace_csv),
        "--output-json",
        str(artifacts.summary_json),
    ]
    run_command(command, cwd=ROOT, timeout=300)
    return artifacts


ARTIFACTS = {config.run_id: run_simulation(config) for config in RUNS}
for run_id, artifacts in ARTIFACTS.items():
    print(f"{run_id}: {artifacts.trace_csv}")
    print(f"{run_id}: {artifacts.summary_json}")



## 3. Cache scalar traces as Parquet

Each trace row gets run metadata so multiple seeds, topologies, action parameters, chains, or proposal diagnostics can be compared without relying on directory names.

In [ ]:
def require_columns(table: pl.DataFrame, required: set[str], *, source: Path) -> None:
    missing = sorted(required.difference(table.columns))
    if missing:
        message = f"{source} is missing required columns: {missing}"
        raise ValueError(message)


def read_trace_table(config: RunConfig, artifacts: RunArtifacts) -> pl.DataFrame:
    if not artifacts.trace_csv.is_file():
        message = f"trace CSV not found: {artifacts.trace_csv}"
        raise FileNotFoundError(message)
    trace = pl.read_csv(artifacts.trace_csv).with_columns(
        pl.col("accepted").cast(pl.Boolean),
        pl.col("proposed").cast(pl.Boolean),
        pl.lit(config.run_id).alias("run_id"),
        pl.lit(config.seed).alias("seed"),
        pl.lit(config.topology).alias("topology"),
        pl.lit(config.cosmological_constant).alias("cosmological_constant"),
        pl.lit(config.temperature).alias("temperature"),
    )
    require_columns(
        trace,
        {"run_id", "step", "accepted", "proposed", "log_prob", "action", "vertices", "edges", "triangles", "move_family"},
        source=artifacts.trace_csv,
    )
    return trace


trace_tables = [read_trace_table(config, ARTIFACTS[config.run_id]) for config in RUNS]
trace_cache = CACHE_DIR / "scalar_trace.parquet"
trace_table = pl.concat(trace_tables, how="vertical")
trace_table.write_parquet(trace_cache)
cached_trace = pl.read_parquet(trace_cache)

print(f"wrote local cache: {trace_cache}")
cached_trace.select(
    pl.len().alias("rows"),
    pl.col("run_id").n_unique().alias("runs"),
    pl.col("accepted").mean().alias("overall_acceptance"),
    pl.col("action").mean().alias("mean_action"),
)



## 4. Cache run summaries and counters

`summary.json` carries run-level configuration, aggregate diagnostics, final counts, move statistics, proposal statistics, and scheduled measurements. The explicitly finite-window effective Hausdorff slope and spectral dimension are nullable here because the Rust API can compute them, but the current summary exporter does not serialize them yet.

In [ ]:
def read_summary(path: Path) -> dict[str, Any]:
    if not path.is_file():
        message = f"summary JSON not found: {path}"
        raise FileNotFoundError(message)
    loaded = json.loads(path.read_text(encoding=UTF8))
    if not isinstance(loaded, dict):
        message = f"summary JSON should be an object: {path}"
        raise TypeError(message)
    return cast("dict[str, Any]", loaded)


def parse_object(value: Any, *, name: str) -> dict[str, Any]:
    if isinstance(value, dict):
        return cast("dict[str, Any]", value)
    message = f"{name} should be an object: {value!r}"
    raise TypeError(message)


def parse_int(value: Any, *, name: str) -> int:
    if isinstance(value, int) and not isinstance(value, bool):
        return value
    message = f"{name} should be an integer: {value!r}"
    raise TypeError(message)


def parse_number(value: Any, *, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float):
        message = f"{name} should be numeric: {value!r}"
        raise TypeError(message)
    message = f"{name} should be finite: {value!r}"
    try:
        parsed = float(value)
    except OverflowError as error:
        raise ValueError(message) from error
    if not math.isfinite(parsed):
        raise ValueError(message)
    return parsed


def expect_rejection(action: Callable[[], object], expected: type[Exception], *, case: str) -> None:
    try:
        action()
    except expected:
        return
    message = f"numeric parser accepted invalid {case}"
    raise AssertionError(message)


expect_rejection(lambda: parse_int(value=True, name="self_check.integer"), TypeError, case="boolean integer")
expect_rejection(lambda: parse_number(value=False, name="self_check.number"), TypeError, case="boolean number")
expect_rejection(lambda: parse_number(value=10**400, name="self_check.number"), ValueError, case="overflowing integer")
for non_finite in (math.nan, math.inf, -math.inf):
    expect_rejection(
        lambda value=non_finite: parse_number(value, name="self_check.number"),
        ValueError,
        case=f"non-finite number {non_finite!r}",
    )


def parse_str(value: Any, *, name: str) -> str:
    if isinstance(value, str):
        return value
    message = f"{name} should be a string: {value!r}"
    raise TypeError(message)


def parse_numeric_list(value: Any, *, name: str) -> list[float]:
    if not isinstance(value, list):
        message = f"{name} should be a list: {value!r}"
        raise TypeError(message)
    return [parse_number(item, name=f"{name}[{index}]") for index, item in enumerate(value)]


def optional_number(mapping: dict[str, Any], keys: tuple[str, ...], *, name: str) -> float | None:
    for key in keys:
        if key in mapping:
            return parse_number(mapping[key], name=f"{name}.{key}")
    return None


def run_summary_row(config: RunConfig, summary: dict[str, Any]) -> RunSummaryRow:
    config_data = parse_object(summary.get("config"), name="summary.config")
    aggregate = parse_object(summary.get("aggregate"), name="summary.aggregate")
    final = parse_object(summary.get("final_triangulation"), name="summary.final_triangulation")
    average_profile = parse_numeric_list(aggregate.get("average_slab_triangle_profile"), name="aggregate.average_slab_triangle_profile")
    fluctuations = parse_numeric_list(aggregate.get("slab_triangle_fluctuations"), name="aggregate.slab_triangle_fluctuations")
    hausdorff = optional_number(aggregate, ("all_scale_effective_hausdorff_slope",), name="aggregate")
    spectral = optional_number(aggregate, ("short_time_effective_spectral_dimension",), name="aggregate")
    return {
        "run_id": config.run_id,
        "seed": parse_int(config_data.get("seed"), name="config.seed"),
        "topology": parse_str(config_data.get("topology"), name="config.topology"),
        "cosmological_constant": parse_number(config_data.get("cosmological_constant"), name="config.cosmological_constant"),
        "temperature": parse_number(config_data.get("temperature"), name="config.temperature"),
        "steps": parse_int(config_data.get("steps"), name="config.steps"),
        "thermalization_steps": parse_int(config_data.get("thermalization_steps"), name="config.thermalization_steps"),
        "measurement_frequency": parse_int(config_data.get("measurement_frequency"), name="config.measurement_frequency"),
        "acceptance_rate": parse_number(aggregate.get("acceptance_rate"), name="aggregate.acceptance_rate"),
        "average_action": parse_number(aggregate.get("average_action"), name="aggregate.average_action"),
        "elapsed_time_ms": parse_number(aggregate.get("elapsed_time_ms"), name="aggregate.elapsed_time_ms"),
        "measurement_count": parse_int(aggregate.get("measurement_count"), name="aggregate.measurement_count"),
        "step_count": parse_int(aggregate.get("step_count"), name="aggregate.step_count"),
        "final_vertices": parse_int(final.get("vertices"), name="final.vertices"),
        "final_edges": parse_int(final.get("edges"), name="final.edges"),
        "final_triangles": parse_int(final.get("triangles"), name="final.triangles"),
        "final_time_slices": parse_int(final.get("time_slices"), name="final.time_slices"),
        "average_slab_triangle_total": math.fsum(average_profile),
        "max_slab_triangle_fluctuation": max(fluctuations, default=0.0),
        "all_scale_effective_hausdorff_slope": hausdorff,
        "short_time_effective_spectral_dimension": spectral,
        "effective_dimensions_exported": hausdorff is not None or spectral is not None,
    }


def counter_rows(config: RunConfig, summary: dict[str, Any]) -> list[CounterRow]:
    rows: list[CounterRow] = []
    for category in ("move_stats", "proposal_stats"):
        counters = parse_object(summary.get(category), name=f"summary.{category}")
        for counter in sorted(counters):
            rows.append(
                {
                    "run_id": config.run_id,
                    "category": category,
                    "counter": counter,
                    "value": parse_int(counters[counter], name=f"{category}.{counter}"),
                },
            )
    return rows


SUMMARY_BY_RUN = {config.run_id: read_summary(ARTIFACTS[config.run_id].summary_json) for config in RUNS}
summary_rows = [run_summary_row(config, SUMMARY_BY_RUN[config.run_id]) for config in RUNS]
counter_table = pl.DataFrame([row for config in RUNS for row in counter_rows(config, SUMMARY_BY_RUN[config.run_id])])
summary_table = pl.DataFrame(summary_rows)

summary_cache = CACHE_DIR / "run_summary.parquet"
counter_cache = CACHE_DIR / "run_counters.parquet"
summary_table.write_parquet(summary_cache)
counter_table.write_parquet(counter_cache)
cached_summary = pl.read_parquet(summary_cache)
cached_counters = pl.read_parquet(counter_cache)

print(f"wrote local cache: {summary_cache}")
print(f"wrote local cache: {counter_cache}")
print(cached_counters.sort("run_id", "category", "counter"))
cached_summary.select(
    "run_id",
    "temperature",
    "cosmological_constant",
    "acceptance_rate",
    "average_action",
    "final_vertices",
    "final_triangles",
    "all_scale_effective_hausdorff_slope",
    "short_time_effective_spectral_dimension",
    "effective_dimensions_exported",
)



## 5. Flatten slab-triangle profiles

Scheduled measurements and nested per-slab triangle profiles are cached separately. The long-form rows have shape `run_id, seed, topology, cosmological_constant, temperature, step, slice, triangles`, which is convenient for Polars group-by operations and matplotlib plots. These `N₂(t)` counts are distinct from configured spatial-vertex counts `N₀(t)`.

In [ ]:
def measurement_rows(config: RunConfig, summary: dict[str, Any]) -> list[MeasurementRow]:
    measurements = summary.get("measurements")
    if not isinstance(measurements, list):
        message = f"summary measurements should be a list for {config.run_id}"
        raise TypeError(message)

    rows: list[MeasurementRow] = []
    for measurement_index, measurement in enumerate(measurements):
        if not isinstance(measurement, dict):
            message = f"measurement {measurement_index} should be an object"
            raise TypeError(message)
        measurement_data = cast("dict[str, Any]", measurement)
        step = parse_int(measurement_data.get("step"), name=f"measurement {measurement_index} step")
        profile = measurement_data.get("slab_triangle_profile")
        if not isinstance(profile, list):
            message = f"measurement {measurement_index} slab_triangle_profile should be a list"
            raise TypeError(message)
        profile_values = [
            parse_number(volume, name=f"measurement {measurement_index} slab_triangle_profile[{slice_index}]") for slice_index, volume in enumerate(profile)
        ]
        rows.append(
            {
                "run_id": config.run_id,
                "seed": config.seed,
                "topology": config.topology,
                "cosmological_constant": config.cosmological_constant,
                "temperature": config.temperature,
                "step": step,
                "action": parse_number(measurement_data.get("action"), name=f"measurement {measurement_index} action"),
                "vertices": parse_int(measurement_data.get("vertices"), name=f"measurement {measurement_index} vertices"),
                "edges": parse_int(measurement_data.get("edges"), name=f"measurement {measurement_index} edges"),
                "triangles": parse_int(measurement_data.get("triangles"), name=f"measurement {measurement_index} triangles"),
                "slab_triangle_total": math.fsum(profile_values),
            },
        )
    return rows


def slab_triangle_profile_rows(config: RunConfig, summary: dict[str, Any]) -> list[SlabTriangleProfileRow]:
    measurements = summary.get("measurements")
    if not isinstance(measurements, list):
        message = f"summary measurements should be a list for {config.run_id}"
        raise TypeError(message)

    rows: list[SlabTriangleProfileRow] = []
    for measurement_index, measurement in enumerate(measurements):
        measurement_data = parse_object(measurement, name=f"measurement {measurement_index}")
        step = parse_int(measurement_data.get("step"), name=f"measurement {measurement_index} step")
        profile = measurement_data.get("slab_triangle_profile")
        if not isinstance(profile, list):
            message = f"measurement {measurement_index} slab_triangle_profile should be a list"
            raise TypeError(message)
        for slice_index, triangles in enumerate(profile):
            rows.append(
                {
                    "run_id": config.run_id,
                    "seed": config.seed,
                    "topology": config.topology,
                    "cosmological_constant": config.cosmological_constant,
                    "temperature": config.temperature,
                    "step": step,
                    "slice": slice_index,
                    "triangles": parse_number(triangles, name=f"measurement {measurement_index} slab {slice_index} triangles"),
                },
            )
    return rows


measurement_rows_all = [row for config in RUNS for row in measurement_rows(config, SUMMARY_BY_RUN[config.run_id])]
slab_triangle_rows = [row for config in RUNS for row in slab_triangle_profile_rows(config, SUMMARY_BY_RUN[config.run_id])]
measurement_schema = {
    "run_id": pl.String,
    "seed": pl.Int64,
    "topology": pl.String,
    "cosmological_constant": pl.Float64,
    "temperature": pl.Float64,
    "step": pl.Int64,
    "action": pl.Float64,
    "vertices": pl.Int64,
    "edges": pl.Int64,
    "triangles": pl.Int64,
    "slab_triangle_total": pl.Float64,
}
slab_triangle_schema = {
    "run_id": pl.String,
    "seed": pl.Int64,
    "topology": pl.String,
    "cosmological_constant": pl.Float64,
    "temperature": pl.Float64,
    "step": pl.Int64,
    "slice": pl.Int64,
    "triangles": pl.Float64,
}
measurement_table = pl.DataFrame(measurement_rows_all, schema=measurement_schema)
slab_triangle_table = pl.DataFrame(slab_triangle_rows, schema=slab_triangle_schema)
measurement_cache = CACHE_DIR / "measurements.parquet"
slab_triangle_cache = CACHE_DIR / "slab_triangle_profile_long.parquet"
measurement_table.write_parquet(measurement_cache)
slab_triangle_table.write_parquet(slab_triangle_cache)
cached_measurements = pl.read_parquet(measurement_cache)
cached_slab_triangles = pl.read_parquet(slab_triangle_cache)

print(f"wrote local cache: {measurement_cache}")
print(f"wrote local cache: {slab_triangle_cache}")
cached_measurements.group_by("run_id").agg(
    pl.len().alias("measurements"),
    pl.col("action").mean().alias("mean_measured_action"),
    pl.col("slab_triangle_total").mean().alias("mean_slab_triangles"),
)

## 6. Plot cached analysis tables

The plots below read from Parquet caches. Delete the cache directory at any time; rerunning the notebook recreates it from the stable CSV/JSON outputs.

In [ ]:
trace_by_run = cached_trace.with_columns(pl.col("accepted").cast(pl.Int64).alias("accepted_int")).sort("run_id", "step")
slab_triangles_by_slice = cached_slab_triangles.group_by("run_id", "slice").agg(pl.col("triangles").mean().alias("mean_triangles")).sort("run_id", "slice")
run_ids = cached_summary.sort("run_id").get_column("run_id").to_list()

figure, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

for (run_id,), run_trace in trace_by_run.group_by("run_id", maintain_order=True):
    steps = run_trace["step"].to_list()
    accepted = [int(value) for value in run_trace["accepted_int"].to_list()]
    running_acceptance = [sum(accepted[: index + 1]) / (index + 1) for index in range(len(accepted))]
    axes[0, 0].plot(steps, run_trace["action"].to_list(), label=run_id)
    axes[0, 1].plot(steps, run_trace["vertices"].to_list(), label=f"{run_id} vertices")
    axes[0, 1].plot(steps, run_trace["triangles"].to_list(), linestyle="--", label=f"{run_id} triangles")
    axes[0, 2].plot(steps, running_acceptance, label=run_id)

for (run_id,), run_profile in slab_triangles_by_slice.group_by("run_id", maintain_order=True):
    axes[1, 0].plot(run_profile["slice"].to_list(), run_profile["mean_triangles"].to_list(), marker="o", label=run_id)

axes[1, 1].bar(run_ids, cached_summary.sort("run_id").get_column("average_action").to_list())
axes[1, 2].bar(run_ids, cached_summary.sort("run_id").get_column("final_vertices").to_list(), label="vertices")
axes[1, 2].bar(run_ids, cached_summary.sort("run_id").get_column("final_triangles").to_list(), alpha=0.55, label="triangles")

axes[0, 0].set_title("Action trace")
axes[0, 0].set_xlabel("step")
axes[0, 0].set_ylabel("Regge action")

axes[0, 1].set_title("Volume counts")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("count")

axes[0, 2].set_title("Running acceptance")
axes[0, 2].set_xlabel("step")
axes[0, 2].set_ylabel("accepted / steps")

axes[1, 0].set_title("Mean slab-triangle profile")
axes[1, 0].set_xlabel("time slice")
axes[1, 0].set_ylabel("triangles")

axes[1, 1].set_title("Average measured action")
axes[1, 1].set_xlabel("run")
axes[1, 1].set_ylabel("action")

axes[1, 2].set_title("Final triangulation counts")
axes[1, 2].set_xlabel("run")
axes[1, 2].set_ylabel("count")

for axis in axes.ravel():
    axis.legend()
    axis.grid(alpha=0.3)

plt.show()